# Feature Engineering

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, make_scorer
from sklearn.pipeline import Pipeline

In [3]:
DATA_DIR = r"D:\Ameng\Data Science Project\heart-failure-prediction\data"

df = pd.read_csv(os.path.join(DATA_DIR, "heart_failure_clinical_records_dataset.csv"))

In [4]:
df.head()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [5]:
def add_egfr(df_new):
    df_new = df_new.copy()
    scr, age, sex = df_new["serum_creatinine"], df_new["age"], df_new["sex"]
    k = np.where(sex == 0, 0.7, 0.9)
    alpha = np.where(sex == 0, -0.241, -0.302)
    df_new["egfr"] = (142 * np.minimum(scr / k, 1.0) ** alpha * np.maximum(scr / k, 1.0) ** -1.200 * 0.9938 ** age)
    return df_new

def add_ef_group(df_new):
    df_new = df_new.copy()
    df_new["ef_group"] = np.where(df_new["ejection_fraction"] < 40, 0, np.where(df_new["ejection_fraction"] <= 49, 1, 2))
    return df_new

In [ ]:
raw_tr_fe = add_ef_group(add_egfr(raw_train))
raw_te_fe = add_ef_group(add_egfr(raw_test))
print("eGFR range :", raw_tr_fe["egfr"].min().round(1), "-", raw_tr_fe["egfr"].max().round(1))
print(raw_tr_fe["ef_group"].value_counts().sort_index())

In [ ]:
raw_tr_fe

In [ ]:
pre_fe = ColumnTransformer([
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one"), ["creatinine_phosphokinase", "egfr", "platelets", "time"]),
    ("scale", StandardScaler(), ["age", "ejection_fraction", "serum_sodium"]),
    ("pass", "passthrough", BIN_COLS + ["ef_group"]),          
])

Xtr_fe = pd.DataFrame(pre_fe.fit_transform(raw_tr_fe), columns=pre_fe.get_feature_names_out())
Xte_fe = pd.DataFrame(pre_fe.transform(raw_te_fe), columns=pre_fe.get_feature_names_out())
print(Xtr_fe.shape, Xte_fe.shape)

In [ ]:
Xtr_fe

In [ ]:
for name, X in [("baseline", X_train), ("fe", Xtr_fe)]:
    print(f"{name:9s}", X.shape, "NaN:", X.isnull().sum().sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(raw_tr_fe["egfr"], bins=30, color="steelblue", alpha=0.7)
axes[0].set_title("eGFR raw")
axes[1].hist(Xtr_fe["log__egfr"], bins=30, color="coral", alpha=0.7)
axes[1].set_title("eGFR after log1p")
plt.tight_layout(); plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scorers = {
    "AUC": make_scorer(roc_auc_score, response_method="predict_proba"),
    "F1": make_scorer(f1_score),
}

def cv_results(X, y, model):
    return {m: cross_val_score(model, X, y, cv=cv, scoring=s).mean().round(4)
            for m, s in scorers.items()}

In [ ]:
models = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}

rows = []
for mname, model in models.items():
    for fname, X in [("baseline", X_train), ("fe", Xtr_fe)]:
        r = cv_results(X, y_train, model)
        r |= {"model": mname, "features": fname}
        rows.append(r)

res = pd.DataFrame(rows)
print(res.sort_values("AUC", ascending=False))

In [ ]:
res.pivot(index="model", columns="features", values="AUC").plot(kind="bar", figsize=(8, 4))
plt.ylabel("CV ROC-AUC")
plt.ylim(0, 1)
plt.tight_layout(); plt.show()

From the result above, there is no significant improvement after feature engineering (egfr and ef_group). The AUC difference between the baseline and fe is to small to believe. <br>
<0.01 difference with 299 rows dataset almost certainly noise. F1-score consistently declining in both model. However, the F1-score is prioritized because the data is imbalanced. <br>
Feature engineering adding complexity without any profit. New feature (eGFR replaces creatinine, plus ef_group) adding interpretation complexity.

After this testing for feature engineering process, no need to export new dataset after fe because there is no significant improvement. So for modelling, use the existing <br>
train and test set without exporting after fe. 